> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 12 · Notebook 05 — Observability, the audit chain, reconciliation and approvals

**Sessions:** S9–S10 (Observability: logs, metrics, dashboards · Reconciliation, audit & approvals) · [Lesson plan](../../docs/lessons/PART_12_TRADING_PLATFORM_CAPSTONE.md) · graded labs in [`labs/part12/`](../../labs/part12/)

**You will:**
1. Log with correlation ids and expose Prometheus metrics; evaluate alert rules and SLOs.
2. Make history tamper-evident with a hash-chained audit log.
3. Reconcile positions and orders with the broker, which is always right.
4. Require a second person for risky changes (four eyes).

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
Everything runs offline on synthetic data with known answers (planted violations, injected faults, a strategy that stops working): no broker, Docker or network is needed.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p12lib.py is in notebooks/part12/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p12lib as p

p.use_course_style()

## 1. Logs, metrics, alerts

Rule of the week: if it isn't logged, measured and reconciled, it didn't happen. Structured JSON logs carry a **correlation id** that follows one idea from signal to fill; metrics are counters, gauges and histograms that Prometheus scrapes:

In [ ]:
from prometheus_client import CollectorRegistry

clock = iter(range(100))
logger = p.StructuredLogger(now=lambda: next(clock))
for msg, fields in (("signal", {"strategy": "mom", "symbol": "SPY", "side": "BUY"}), ("intent", {"qty": 100}),
                    ("order", {"order_id": "O17", "limit": 512.31}), ("fill", {"order_id": "O17", "price": 512.30})):
    logger.log("info", msg, cid="c-7f3a", **fields)
logger.log("info", "heartbeat", cid="c-0000")
print("the story of one trade:", [r["msg"] for r in logger.trace("c-7f3a")])

registry = CollectorRegistry()
metrics = p.make_metrics(registry)
metrics["orders"].labels("mom", "ib").inc(3)
metrics["rejects"].labels("ib").inc()
for x in p.latency_samples()[:50]:
    metrics["latency"].observe(x)
metrics["staleness"].labels("polygon").set(2.4)
print("\n".join(line for line in p.exposition(registry).splitlines() if line.startswith(("qf_orders_total{", "qf_rejects_total{", "qf_signal_to_submit_seconds_count", "qf_data_staleness"))))

rules = [("data stale", lambda s: s["staleness_s"] > 5, "critical"), ("reject rate high", lambda s: s["reject_rate"] > 0.05, "warning"),
         ("latency p99 high", lambda s: s["p99_s"] > 0.1, "warning"), ("disk low", lambda s: s["disk_free"] < 0.15, "info")]
print("\nalerts firing:", p.evaluate_alerts({"staleness_s": 8, "reject_rate": 0.25, "p99_s": 0.08, "disk_free": 0.1}, rules))

An **SLO** says "the value stays under a threshold for a target share of the time". Report the compliance (share of samples at or under the threshold), whether the target is met, and how much of the **error budget** is used: the share of bad samples divided by the allowed share `1 − target` (1.0 = the whole budget spent).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def slo_report(values, threshold, target=0.999):
    ok = float((values <= threshold).mean())
    return {"compliance": ok, "met": ok >= target, "budget_used": (1 - ok) / (1 - target)}

lat = pd.Series(p.latency_samples())
cases = [(lat, 0.1), (lat, 0.06, 0.99), (lat, 0.05, 0.9), (pd.Series([0.01, 0.02, 0.2, 0.03]), 0.1, 0.75)]
mine = [p.attempt(slo_report, *c) for c in cases]
mine = p.check("slo_report", mine, [p.slo_report(*c) for c in cases])
pd.DataFrame(mine, index=["ack ≤ 100 ms, 99.9%", "ack ≤ 60 ms, 99%", "ack ≤ 50 ms, 90%", "toy"]).round(3)

One 400 ms spike in 300 acks already spends more than the whole 99.9% budget: tight SLOs make rare problems visible.

## 2. A hash-chained audit log

Every record stores the previous record's hash (`prev`) and its own `hash` = sha256 of the record without its hash (`p._digest(body)`). To verify, walk the chain: a record is broken if its `prev` isn't the previous hash, or its hash doesn't match its body. Return the index of the first broken record, −1 if intact.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
class MyAuditLog(p.AuditLog):
    def verify(self):
        prev = "0" * 64
        for i, rec in enumerate(self.records):
            body = {k: v for k, v in rec.items() if k != "hash"}
            if rec["prev"] != prev or p._digest(body) != rec["hash"]:
                return i
            prev = rec["hash"]
        return -1

def filled_log(cls):
    clock = iter(range(1000))
    log = cls(now=lambda: next(clock))
    for i in range(6):
        log.append("order", id=f"O{i}", qty=100)
        log.append("fill", order_id=f"O{i}", qty=100, price=100 + i)
    return log

intact, edited, relinked = filled_log(MyAuditLog), filled_log(MyAuditLog), filled_log(MyAuditLog)
edited.records[5]["data"]["price"] = 99                              # someone "fixes" a fill price
relinked.records[5]["data"]["price"] = 99
relinked.records[5]["hash"] = p._digest({k: v for k, v in relinked.records[5].items() if k != "hash"})   # and re-hashes it
mine = p.check("AuditLog.verify", [p.attempt(log.verify) for log in (intact, edited, relinked)],
               [p.AuditLog.verify(log) for log in (intact, edited, relinked)])
print(f"intact chain: {mine[0]};  edited record 5: breaks at {mine[1]};  edited and re-hashed: breaks at {mine[2]} (the NEXT record's link)")

Editing a record breaks its hash; re-hashing it breaks the next record's link, and so on to the end. Sending the latest hash out every day (the `head()`) means even rewriting the whole tail can be detected.

## 3. Reconciliation

Compare our positions with the broker's over the union of symbols (missing = 0) and list every break: `{symbol, internal, broker, diff = broker − internal}`, sorted by symbol. The **broker** is the source of truth.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def reconcile_positions(internal, broker, tol=1e-9):
    out = []
    for s in sorted(set(internal) | set(broker)):
        a, b = internal.get(s, 0.0), broker.get(s, 0.0)
        if abs(a - b) > tol:
            out.append({"symbol": s, "internal": a, "broker": b, "diff": b - a})
    return out

cases = [({"SPY": 300, "QQQ": -100}, {"SPY": 300, "QQQ": -100}), ({"SPY": 300, "QQQ": -100}, {"SPY": 200, "QQQ": -100, "IWM": 50}),
         ({"SPY": 100}, {})]
mine = [p.attempt(reconcile_positions, *c) for c in cases]
mine = p.check("reconcile_positions", mine, [p.reconcile_positions(*c) for c in cases])
pd.DataFrame(mine[1])

## 4. Four eyes

New strategies, parameter and limit changes, resuming a paused strategy and large orders need a second person. `p.ApprovalQueue` audits every step and refuses self-approval:

In [ ]:
audit = p.AuditLog()
approvals = p.ApprovalQueue(audit)
aid = approvals.request("limit_change", {"max_gross": 1.5}, requester="alice", reason="new pairs book")
try:
    approvals.decide(aid, approver="alice", approve=True, reason_code="ok")
except ValueError as err:
    print("alice approving her own request:", err)
approvals.decide(aid, approver="bob", approve=True, reason_code="reviewed_backtest")
print(approvals.items[aid]["status"], "| audit:", [r["event"] for r in audit.records], "| chain intact:", audit.verify() == -1)
print("order of 5,000 × $120 needs a human:", p.needs_approval({"qty": 5000, "price": 120.0}, max_notional=250_000))

## Wrap-up

* Correlation ids tell the story of a trade; metrics feed alerts and SLOs with explicit error budgets.
* The audit log is append-only and hash-chained; publish its head daily.
* Reconcile with the broker; the broker wins; every break is explained.
* Four eyes on anything that changes risk.
* Graded version: `labs/part12/week43_monitoring` (`StructuredLogger`, `make_metrics`, `evaluate_alerts`, `slo_report`, `AuditLog`, `reconcile_positions`, `reconcile_orders`, `ApprovalQueue`).